# 02｜監督式學習實作：臺北市房價預測

**高一學生實作版｜只使用一份 `Taipei_house.csv`**

本週以實際交易紀錄練習監督式學習。資料共 **12,429 筆、16 欄**；使用建物總面積等特徵（X）預測交易總價（y），並學習 LinearRegression、MAE、R²。

### 課前準備

1. 在 Google Colab **執行第一題之前**，請先把教師提供的 `Taipei_house.csv` 上傳到執行環境的檔案區。若使用 VS Code，請將 CSV 放在 Notebook 的工作資料夾。
2. 每一題先讀 **Markdown 參考範例**，再於**下一個空白 Code cell** 自己輸入，按 **Shift + Enter** 執行。
3. 請依照順序練習；後面的程式會沿用前面建立的 `house`、`X_train`、`model` 等變數。重新連接 Colab 後，CSV 可能需要再次上傳。
4. 需要 `pandas`、`numpy`、`matplotlib`、`scikit-learn`，Colab 通常已提供。

> **資料限制：** 這份 CSV 只包含臺北市部分行政區、部分年份的交易；「總價」及「建物總面積」都沒有隨附明確計價／面積單位，因此課堂統一稱為**原始資料單位**，不自行換算為萬元或坪。房價模型僅作教學練習，不是個別房屋估價。

---
## 第一節｜資料與監督式學習

使用同一份臺北市房價 CSV，認識資料列、特徵 X、標籤 y、散佈圖，最後切成訓練與測試資料。

### 練習 1-1｜讀取房價 CSV

使用 `pd.read_csv()` 讀取檔案；`shape` 會顯示資料筆數與欄數。若出現 `FileNotFoundError`，請先將 `Taipei_house.csv` 上傳 Colab。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
import pandas as pd

house = pd.read_csv("Taipei_house.csv")
print("資料形狀：", house.shape)
print(house.head(3))
```

**輪到你：** 將 `head(3)` 改成 `head(5)`。

### 練習 1-2｜找出特徵與標籤

一列代表一筆房屋交易紀錄。`建物總面積` 是可用於預測的特徵，`總價` 是希望預測的標籤。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
print(house.columns.tolist())
print(house[["行政區", "建物總面積", "總價"]].head())
```

**輪到你：** 把 `屋齡` 加入顯示欄位。

### 練習 1-3｜認識回歸與分類

預測**總價數值**是回歸；若預測「高價／低價」這種種類，才是分類。本課使用回歸任務。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
print("房價總價範例：", house["總價"].head().tolist())
print("資料筆數：", len(house))
```

**輪到你：** 說明為什麼 `總價` 適合用回歸預測，而不是分類。

### 練習 1-4｜資料篩選：觀察行政區

真實資料通常有不同群組。先檢查有哪些行政區，再篩選特定行政區的紀錄。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
print(house["行政區"].value_counts())
wen = house[house["行政區"] == "文山區"]
print("文山區筆數：", len(wen))
```

**輪到你：** 換成 `信義區`，比較交易筆數。

### 練習 1-5｜用散佈圖觀察面積與總價

橫軸是建物總面積，縱軸是總價。散點圖可能存在極端值，先抽出 600 筆讓圖較容易閱讀。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
import matplotlib.pyplot as plt

part = house.sample(600, random_state=42)
plt.scatter(part["建物總面積"], part["總價"], s=8, alpha=0.35)
plt.xlabel("Building area (original unit)")
plt.ylabel("Total price (original unit)")
plt.show()
```

**輪到你：** 觀察是否所有房屋面積越大，總價就一定越高。

### 練習 1-6｜建立 X 和 y

`X` 必須保留**二維**（`[[...]]`），`y` 則取出一個數值欄位（`[...]`）。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
X = house[["建物總面積"]]
y = house["總價"]
print("X：", X.shape)
print("y：", y.shape)
```

**輪到你：** 解釋為什麼 X 的 shape 第二個數字是 1。

### 練習 1-7｜把資料切成訓練和測試

將資料切成 80% 訓練、20% 測試。`random_state=42` 使上課的分組結果一致。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("訓練：", len(X_train))
print("測試：", len(X_test))
```

**輪到你：** 想一想為什麼要保留測試集，不能拿它的答案先訓練。

### 練習 1-8｜檢查訓練與測試資料是否分開

兩份資料不能含有同一筆交易紀錄。使用 Pandas 的 `index` 來確認。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
same = X_train.index.intersection(X_test.index)
print("重複筆數：", len(same))
print("訓練占比：", round(len(X_train) / len(house), 2))
```

**輪到你：** 推測若把同一筆資料放進訓練及測試，會影響什麼？

---
## 第二節｜LinearRegression：訓練與預測

用「建物總面積」建立最簡單的線性模型，理解 fit、predict、迴歸線、殘差。

### 練習 2-1｜建立線性迴歸模型

`LinearRegression()` 是回歸模型，`fit()` 會從訓練資料找出適合的直線。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
print("模型已訓練")
```

**輪到你：** 指出 `fit()` 的 X 和 y 各代表什麼。

### 練習 2-2｜學出的斜率與截距

單特徵線性迴歸可寫成 `預測總價 = 斜率 × 建物總面積 + 截距`。係數是資料學出的，不是自己猜的。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
print("斜率：", model.coef_[0])
print("截距：", model.intercept_)
```

**輪到你：** 說明斜率代表建物總面積增加 1 個原始單位，模型預測總價會如何改變。

### 練習 2-3｜把模型直線畫在散佈圖上

這張圖的直線由模型係數計算，點則是交易資料。**線不是每個點都要穿過。**

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
import numpy as np

part = house.sample(600, random_state=42)
plt.scatter(part["建物總面積"], part["總價"], s=8, alpha=0.3)
a = np.array([0, 400])
b = model.coef_[0] * a + model.intercept_
plt.plot(a, b, color="red")
plt.xlabel("Building area")
plt.ylabel("Total price")
plt.show()
```

**輪到你：** 解釋為什麼有的點在直線上方，有的在下方。

### 練習 2-4｜預測一筆新房屋

新資料的欄位名稱和訓練資料必須相同。這只是模型估計，並不是實際交易總價。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
new = pd.DataFrame({"建物總面積": [120]})
print("預測總價：", model.predict(new)[0])
```

**輪到你：** 把面積改成 150，比較兩次預測結果。

### 練習 2-5｜預測整份測試集

`predict(X_test)` 會對每筆測試資料產生一個預測總價，再和 `y_test` 比較。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
pred = model.predict(X_test)
print("前五筆預測：", pred[:5])
print("前五筆真實：", y_test.iloc[:5].tolist())
```

**輪到你：** 比較第一筆測試資料的預測值與真實值。

### 練習 2-6｜殘差：真實總價減去預測總價

殘差（residual）=`真實值 − 預測值`；正值表示模型低估、負值表示模型高估。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
error = y_test.to_numpy() - pred
print("前五筆殘差：", error[:5])
print("第一筆絕對誤差：", abs(error[0]))
```

**輪到你：** 把第一筆殘差轉成正負號的文字解釋。

### 練習 2-7｜建立預測與真實值對照表

利用 Pandas 把測試資料的真實總價、預測總價放在同一張表。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
result = pd.DataFrame({
    "真實總價": y_test.to_numpy(),
    "預測總價": pred
})
print(result.head())
```

**輪到你：** 新增一欄 `"誤差"`，計算真實總價減預測值。

### 練習 2-8｜改變一筆輸入，觀察模型預測

同一個訓練完成的模型，輸入不同建物面積，得到不同的總價估計。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
more = pd.DataFrame({"建物總面積": [80, 120, 160]})
print(model.predict(more))
```

**輪到你：** 再試一筆建物總面積 200 的房屋。

---
## 第三節｜評估與改善房價模型

以固定的訓練／測試集比較 MAE、R²；再試著加入房數、屋齡與電梯特徵。

### 練習 3-1｜MAE：平均相差多少總價

MAE（平均絕對誤差）是 `|真實 − 預測|` 的平均。越低通常表示平均誤差越小，單位與 CSV 的總價相同。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, pred)
print("測試 MAE：", round(mae, 2))
```

**輪到你：** 解釋 MAE 為 865.53 時表示什麼（請使用原始總價單位）。

### 練習 3-2｜R²：與猜平均值相比

R² 幫我們看模型能解釋多少測試資料的變動；越接近 1 通常越好，也可能小於 0。R² 不是 Accuracy。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
from sklearn.metrics import r2_score

r2 = r2_score(y_test, pred)
print("測試 R²：", round(r2, 3))
```

**輪到你：** 比較 R²=0.7 與 R²=0.4，哪個表示此指標較好？

### 練習 3-3｜訓練與測試誤差

要觀察是否只對訓練資料比較準。這是一次切分的結果，不能用來保證其他資料的成績。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
train_pred = model.predict(X_train)
print("訓練 MAE：", mean_absolute_error(y_train, train_pred))
print("測試 MAE：", mean_absolute_error(y_test, pred))
```

**輪到你：** 比較訓練和測試 MAE，討論兩者為何有差異。

### 練習 3-4｜檢查真實資料品質

真實資料沒有缺失值不等於沒有特殊數值；這份 CSV 中部分屋齡小於 0，應保留並討論來源定義。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
print("缺失值數量：", house.isna().sum().sum())
print("負屋齡筆數：", (house["屋齡"] < 0).sum())
print(house["總價"].describe())
```

**輪到你：** 想想不合理或特殊屋齡可能來自什麼資料定義，不要自行假設。

### 練習 3-5｜增加「房數」比較模型

使用**同一筆切分**，將單一特徵改成兩個特徵，再與前面的 MAE、R² 比較。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
cols = ["建物總面積", "房數"]
two = LinearRegression()
two.fit(house.loc[X_train.index, cols], y_train)
two_pred = two.predict(house.loc[X_test.index, cols])
print("兩特徵 MAE：", round(mean_absolute_error(y_test, two_pred), 2))
print("兩特徵 R²：", round(r2_score(y_test, two_pred), 3))
```

**輪到你：** 哪個模型在這份測試集上的 MAE 較低？

### 練習 3-6｜再加入屋齡與電梯

四個數值特徵的模型一樣使用固定的資料切分。注意：有些 `屋齡` 是負數，這裡只是為了示範比較而保留。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
cols4 = ["建物總面積", "屋齡", "房數", "電梯"]
four = LinearRegression()
four.fit(house.loc[X_train.index, cols4], y_train)
four_pred = four.predict(house.loc[X_test.index, cols4])
print("四特徵 MAE：", round(mean_absolute_error(y_test, four_pred), 2))
print("四特徵 R²：", round(r2_score(y_test, four_pred), 3))
```

**輪到你：** 增加特徵一定會讓每筆預測都變準嗎？

### 練習 3-7｜比較三種模型的測試成績

模型要公平比較，就維持相同資料切分與指標。MAE 越低越好，R² 越高越好。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
summary = pd.DataFrame({
    "模型": ["面積", "面積+房數", "四特徵"],
    "MAE": [mean_absolute_error(y_test, p) for p in [pred, two_pred, four_pred]],
    "R2": [r2_score(y_test, p) for p in [pred, two_pred, four_pred]]
})
print(summary.round(3))
```

**輪到你：** 根據表格用兩句話解釋哪一組特徵在這份測試集表現較好。

### 練習 3-8｜用兩特徵預測新房屋

用訓練好的兩特徵模型估計「建物總面積 120、房數 3」的交易總價。注意 DataFrame 欄名與順序。

**參考程式（先閱讀，再在下方空白 Code cell 輸入）：**

```python
new = pd.DataFrame({"建物總面積": [120], "房數": [3]})
print("兩特徵預測：", two.predict(new)[0])
print("本次兩特徵測試 MAE：", round(mean_absolute_error(y_test, two_pred), 2))
```

**輪到你：** 把房數從 3 改成 4，再用一句話提醒這只是模型估計。

---
## 完成後，你應該能夠……

- 使用 Pandas 讀取 **`Taipei_house.csv`**，說明資料列、特徵 X、標籤 y。
- 使用散佈圖理解建物總面積與總價的關係，以及其限制。
- 使用 `train_test_split`、`LinearRegression` 的 `fit()`、`predict()`。
- 看懂 MAE、R²、訓練與測試誤差。
- 保留相同切分，比較一特徵、兩特徵、四特徵模型。
- 注意資料來源、屋齡特殊值與原始總價單位未知的限制。

**交作業前：** 確認 24 個練習 Code cell 已自行輸入並執行；全程只需一份 **`Taipei_house.csv`**。